## 1. Raw Data
Loading the F1 Pit Strategy dataset.

In [ ]:
import pandas as pd
import numpy as np
import os

# Load data
data_path = 'playground-series-s6e5' if os.path.exists('playground-series-s6e5') else '.'
train = pd.read_csv(f'{data_path}/train.csv')
test = pd.read_csv(f'{data_path}/test.csv')
print(f'Train shape: {train.shape}')
print(f'Test shape: {test.shape}')

## 2. Data Validation
Basic checks for consistency.

In [ ]:
print('Missing values in Train:')
print(train.isnull().sum())
print('\nData Types:')
print(train.dtypes)

## 3. Exploratory Data Analysis (EDA)
Visualizing the relationship between new features and the target variable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Feature calculation for EDA
train['PitWindowPressure'] = train['TyreLife'] * train['RaceProgress']

# Plot: PitWindowPressure distribution by PitNextLap (Box Plot)
plt.figure(figsize=(10, 6))
sns.boxplot(x='PitNextLap', y='PitWindowPressure', data=train)
plt.title('PitWindowPressure distribution by PitNextLap')
plt.show()

# Plot: Cumulative_Degradation distribution by PitNextLap (KDE plot)
plt.figure(figsize=(10, 6))
sns.kdeplot(data=train, x='Cumulative_Degradation', hue='PitNextLap', fill=True)
plt.title('Cumulative_Degradation distribution by PitNextLap')
plt.show()

## 4. Data Preprocessing & Feature Engineering
Implementing high-value features derived from the slower notebook.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# Clean column names
train.columns = [c.replace(' ', '_').replace('(', '').replace(')', '').replace('[', '').replace(']', '') for c in train.columns]
test.columns = [c.replace(' ', '_').replace('(', '').replace(')', '').replace('[', '').replace(']', '') for c in test.columns]

# Ordinal Encode 'Driver', 'Compound', 'Race'
cat_cols = ['Driver', 'Compound', 'Race']
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
train[cat_cols] = encoder.fit_transform(train[cat_cols].astype(str))
test[cat_cols] = encoder.transform(test[cat_cols].astype(str))

# New Feature: TyreLifeBin
bins = [0, 3, 6, 10, 15, 20, 30, 40, 100]
train['TyreLifeBin'] = pd.cut(train['TyreLife'], bins=bins, labels=False)
test['TyreLifeBin'] = pd.cut(test['TyreLife'], bins=bins, labels=False)

# New Feature: PitWindowPressure
train['PitWindowPressure'] = train['TyreLife'] * train['RaceProgress']
test['PitWindowPressure'] = test['TyreLife'] * test['RaceProgress']

# New Feature: Deg_diff
train['Deg_diff'] = train.groupby(['Driver', 'Stint'])['Cumulative_Degradation'].diff().fillna(0)
test['Deg_diff'] = test.groupby(['Driver', 'Stint'])['Cumulative_Degradation'].diff().fillna(0)

print('Preprocessing and Feature Engineering complete.')

## 5. Model Training
Using the baseline LightGBM model with the new features.

In [ ]:
import lightgbm as lgb

# Define features and target
X = train.drop(['id', 'PitNextLap'], axis=1)
y = train['PitNextLap']

# LGBM params with GPU support (fallback to CPU)
params = {
    'objective': 'binary',
    'metric': 'auc',
    'verbosity': -1,
    'device': 'gpu',
    'random_state': 42
}

# GPU detection probe
try:
    _probe = lgb.LGBMClassifier(n_estimators=2, device='gpu')
    _probe.fit(np.random.rand(10, X.shape[1]), np.random.randint(0, 2, 10))
    print('Using GPU for LightGBM')
except:
    print('Falling back to CPU for LightGBM')
    params['device'] = 'cpu'

model = lgb.LGBMClassifier(**params)

## 6. Model Evaluation
Hold-out set evaluation.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

eval_model = lgb.LGBMClassifier(**params)
eval_model.fit(X_train, y_train)

val_preds = eval_model.predict_proba(X_val)[:, 1]
auc_score = roc_auc_score(y_val, val_preds)
print(f'ROC-AUC on hold-out set: {auc_score}')

## 7. Model Validation
5-Fold Stratified Cross-Validation.

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
aucs = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_t, X_v = X.iloc[train_idx], X.iloc[val_idx]
    y_t, y_v = y.iloc[train_idx], y.iloc[val_idx]
    
    m = lgb.LGBMClassifier(**params)
    m.fit(X_t, y_t)
    p = m.predict_proba(X_v)[:, 1]
    auc = roc_auc_score(y_v, p)
    aucs.append(auc)
    print(f'Fold {fold+1} AUC: {auc}')

print(f'\nMean AUC: {np.mean(aucs):.5f} +/- {np.std(aucs):.5f}')

## 8. Model Deployment & Feedback
Outputting artifacts.

In [ ]:
import pickle

# Final model on full data
final_model = lgb.LGBMClassifier(**params)
final_model.fit(X, y)

# Save the model
with open('lgbm_experiment2.pkl', 'wb') as f:
    pickle.dump(final_model, f)

# Generate submission
test_X = test.drop(['id'], axis=1)
test_preds = final_model.predict_proba(test_X)[:, 1]
submission = pd.DataFrame({'id': test['id'], 'PitNextLap': test_preds})
submission.to_csv('submission_exp2.csv', index=False)
print('Saved lgbm_experiment2.pkl and submission_exp2.csv')